# RetailLens — Reproducible Temporal Validation

**Purpose:** Reproduce key reported analyses using *saved customer-level validation checkpoints* from the Online Retail II project. Designed for students and technical reviewers.

This is **not** the full pipeline from the raw Excel workbook. The earlier exploratory notebook contains the initial data audits, matching logic, and construction of the historical customer tables. The CSV inputs here contain only anonymized RFM scores, segment labels, and 90-day outcomes (not transaction-level records or customer IDs).

**Question:** Can the RFM score discriminate between previously observed customers who do and do not make a qualifying purchase in the following 90 days?

**Definitions:** R, F, M are 1–5 ordinal scores; higher is better. `Repeat90d=1` denotes at least one eligible positive purchase line in the next 90 days. `Repeat90d_adj` removes exactly matched reversals known within the September outcome window; it is *not* complete refund accounting.

**Requirements:** `pandas`, `numpy`, `matplotlib`, `scikit-learn`. Start Jupyter in the **repository root**, which contains the `data/` folder. In Colab, clone the full repository first or upload both CSV files into a `data/` folder alongside the notebook's working directory.

## 1. Load saved validation cohorts
Each row is a historical customer; no customer IDs are included in the published checkpoint files.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score

base = next((p for p in [Path.cwd(), Path.cwd().parent]
             if (p / 'data/june_validation.csv').exists()), None)
assert base is not None, 'Run from repo root or its notebooks folder.'
june = pd.read_csv(base / 'data/june_validation.csv')
sept = pd.read_csv(base / 'data/september_validation.csv')
print('June customers:', len(june), '| September:', len(sept))

## 2. Check data quality and denominators
Assertions deliberately stop if files have unexpected or missing columns; never interpret ROC-AUC without checking the target has both classes.

In [ ]:
cols = ['R_score', 'F_score', 'M_score', 'Segment', 'Repeat90d']
for cohort in [june, sept]:
    assert not cohort[cols].isna().any().any()
    assert cohort['Repeat90d'].isin([0, 1]).all()
    assert cohort['Repeat90d'].nunique() == 2
    assert cohort[['R_score','F_score','M_score']].isin(range(1, 6)).all().all()
assert len(june) == 4938 and len(sept) == 5251
assert june['Repeat90d'].sum() == 1581 and sept['Repeat90d'].sum() == 2286
print('QA passed.')

## 3. Compare Recency-only with combined RFM
The combined RFM score is **R_score + F_score + M_score** with equal weights. ROC-AUC measures ranking discrimination, **not** classification accuracy.

In [ ]:
def auc_results(df):
    y = df['Repeat90d']
    r = df['R_score']
    rfm = df[['R_score', 'F_score', 'M_score']].sum(axis=1)
    return [roc_auc_score(y, r), roc_auc_score(y, rfm)]

results = pd.DataFrame(
    [auc_results(june), auc_results(sept)],
    columns=['Recency only', 'Combined RFM'],
    index=['June 2011', 'September 2011'])
results['RFM improvement'] = results['Combined RFM'] - results['Recency only']
print(results.round(4))

## 4. Check improvement uncertainty with paired bootstrap
Customers are sampled with replacement and each scoring method uses the **same sample**. The interval describes sampling uncertainty under the bootstrap assumptions; it is not external validation.

In [ ]:
def bootstrap_gain(df, left, right, n=500, seed=42):
    y = df['Repeat90d'].to_numpy()
    a, b = np.asarray(left), np.asarray(right)
    rng, gains = np.random.default_rng(seed), []
    for _ in range(n):
        i = rng.integers(0, len(df), len(df))
        if len(np.unique(y[i])) == 2:
            gains.append(roc_auc_score(y[i], a[i]) - roc_auc_score(y[i], b[i]))
    return np.percentile(gains, [2.5, 97.5])

for period, df in [('June', june), ('September', sept)]:
    total = df[['R_score','F_score','M_score']].sum(axis=1)
    ci = bootstrap_gain(df, total, df['R_score'])
    print(period, 'RFM minus Recency 95% bootstrap CI:', ci.round(4))

## 5. Compare the eight segments over time
These rates are descriptive. Segment boundaries were defined from historical scores; labels are not fitted probability models.

In [ ]:
segment_rates = pd.DataFrame({
    'June 2011': june.groupby('Segment')['Repeat90d'].mean() * 100,
    'September 2011': sept.groupby('Segment')['Repeat90d'].mean() * 100
}).sort_values('September 2011')
print(segment_rates.round(2))
ax = segment_rates.plot.barh(figsize=(10, 6), width=0.75)
ax.set(xlim=(0, 100), xlabel='Observed 90-day repeat-purchase rate (%)',
       ylabel='', title='90-Day Repeat-Purchase Rates by Segment')
plt.tight_layout()
plt.show()

## 6. Out-of-time segment ranking
Use **June** segment outcome rates to rank **September** customers. Do not calculate those rates from September labels, as that would evaluate a score informed by the same future outcomes.

In [ ]:
june_rates = june.groupby('Segment')['Repeat90d'].mean()
segment_score = sept['Segment'].map(june_rates)
assert segment_score.notna().all()
segment_auc = roc_auc_score(sept['Repeat90d'], segment_score)
print('September segment AUC using June rates:', round(segment_auc, 4))
print('September combined RFM AUC:', round(results.loc['September 2011','Combined RFM'], 4))

## 7. Bootstrap the segment comparisons
Compare segment ranking against Recency and combined RFM on the **same September customers**.

In [ ]:
sept_total = sept[['R_score','F_score','M_score']].sum(axis=1)
for name, baseline in [('Recency', sept['R_score']), ('Combined RFM', sept_total)]:
    ci = bootstrap_gain(sept, segment_score, baseline)
    print('Segments minus', name, '95% bootstrap CI:', ci.round(4))

## 8. September cancellation-outcome sensitivity
Changing the label for 2 of 5,251 customers should barely change the measured AUC, but this does **not** resolve all refunds or evaluate every cancellation policy.

In [ ]:
assert sept['Repeat90d_adj'].isin([0,1]).all()
changed = (sept['Repeat90d'] != sept['Repeat90d_adj']).sum()
original = roc_auc_score(sept['Repeat90d'], sept_total)
adjusted = roc_auc_score(sept['Repeat90d_adj'], sept_total)
print('Outcomes changed:', changed)
print('Original combined AUC:', round(original, 4))
print('Adjusted combined AUC:', round(adjusted, 4))

## 9. Visualize AUC results
Use a 0–1 y-axis to avoid exaggerating small performance differences.

In [ ]:
ax = results[['Recency only','Combined RFM']].plot.bar(figsize=(8, 5), rot=0)
ax.set(xlabel='Historical cutoff', ylabel='ROC-AUC', ylim=(0,1),
       title='RetailLens: ROC-AUC across two historical cutoffs')
for bars in ax.containers:
    ax.bar_label(bars, fmt='%.4f', padding=3)
plt.tight_layout()
plt.show()

## 10. Interpretation and limitations

- The **combined score** is an equal-weight formula (3–15), not a trained machine-learning classifier or a probability estimate.
- The **eight segments** are ordered, transparent business rules. Their out-of-time segment-rate ranking has lower observed AUC than the combined RFM score in September.
- Scores and labels in these CSVs were **previously computed from raw transactions** using the project's documented cleaning and reversal rules; this notebook **does not reproduce those upstream steps**.
- All results come from a single historical UK retailer; some customers appear in both evaluation cohorts. Two dates are encouraging but do not constitute independent external validation.
- Positive purchase activity does not guarantee an order was retained, and no campaign treatment was randomized. **Do not claim increased revenue, profit, or causal retention uplift.**
- This checkpoint notebook enables reviewers to independently recalculate the stated validation metrics. The full raw-to-feature pipeline requires a separate cleaned notebook and a fresh-runtime end-to-end audit.